In [ ]:

from astropy.io import fits
from astropy.table import Table as t
import numpy as np
import seaborn as sns

import h5py
from astropy.convolution import Gaussian1DKernel, convolve
from collections import defaultdict
import os

from sklearn.neighbors import radius_neighbors_graph
from scipy.sparse.csgraph import connected_components
import networkx as nx
from matplotlib.colors import ListedColormap
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import zoomed_inset_axes
from mpl_toolkits.axes_grid1.inset_locator import mark_inset

os.environ['PATH'] = '/Library/TeX/texbin:' + os.environ['PATH']
plt.rcParams['text.usetex'] = True
plt.style.use('./data/plots/desi.mplstyle')
plt.rcParams['figure.dpi'] = 360

In [ ]:
cmap = sns.color_palette("mako", as_cmap=True)
cmap

# files

In [ ]:
path = './desi_data/10256/20211110/coadd-7-10256-thru20211110.fits'
id = 39627563043129640

In [ ]:
data = fits.open(path)

In [ ]:
data[3].data.shape

In [ ]:
table = t(data[1].data)
table

In [ ]:
spec = table[str(table['TARGETID'])[:1] == str(id)[:1]]
spec

In [ ]:
ra = spec['TARGET_RA'].data[0]
dec = spec['TARGET_DEC'].data[0]
ra, dec

In [ ]:
redr = fits.open('./desi_data/10256/20211110/redrock-8-10256-thru20211110.fits')
redr

In [ ]:
tab = t(redr[1].data)
tab[tab['TARGETID'] == id]

In [ ]:
import requests
from PIL import Image
from io import BytesIO

url = (
    f"https://www.legacysurvey.org/viewer/cutout.jpg?"
    f"ra={ra}&dec={dec}"
    f"&layer=ls-dr10"
    f"&pixscale=0.262"
    f"&bands=grz"
)

resp = requests.get(url)
img = Image.open(BytesIO(resp.content))
plt.show()

In [ ]:
file = np.load('./processed/umap/umap_20211110_10256.npz')
file.files

In [ ]:
file = h5py.File('./processed/20211110-10256-7.h5', 'r')

In [ ]:
file.keys()

In [ ]:
file['spectra']['B']

# umap

In [ ]:
file = np.load('./processed/umap/umap_20211110_10256.npz', allow_pickle=True)

In [ ]:
file.files

In [ ]:
emb, labels, outliers = file['embedding'], file['labels'], file['outlier_mask']
cat, ids, petals = file['categories'], file['ids'], file['petals']

In [ ]:
emb.shape

In [ ]:
plt.plot(emb[:, 0], emb[:, 1], 'o', markersize=6, alpha=0.5)

In [ ]:
cat_ = np.array([c.decode('utf-8') for c in cat])
np.unique(cat_)

In [ ]:
colors = cmap(np.linspace(0.1, 0.9, 3))
colors

In [ ]:
for i, c in enumerate(np.unique(cat_)):
    mask = np.where(cat_ == c)[0]
    plt.plot(emb[mask, 0], emb[mask, 1], 'o', markersize=5, alpha=0.8, label=c, color=colors[i])
#
plt.scatter(emb[outliers, 0], emb[outliers, 1], marker='x', s=60, alpha=1., lw=4, label='outliers',
            c='firebrick', zorder=10)
# plt.scatter(emb[outliers, 0], emb[outliers, 1], marker='o', s=80, alpha=0.6, label='outliers',
#             c='firebrick', zorder=-1)
plt.legend(fontsize=10, loc='upper right')
plt.axis('off')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
nx.draw_networkx_edges(G, pos, alpha=0.5, width=0.1)

for i, c in enumerate(np.unique(cat_)):
    mask = np.where(cat_ == c)[0]
    ax.plot(emb[mask, 0], emb[mask, 1], 'o', markersize=4, alpha=0.8, label=c, color=colors[i])

ax.scatter(emb[outliers, 0], emb[outliers, 1], marker='o', s=120, alpha=0.6, label='outliers',
            c='firebrick', zorder=-1)

axins = zoomed_inset_axes(ax, 1, loc=1)
mark_inset(ax, axins,  loc1=3, loc2=2, fc='none', ec='white', linewidth=0.5)

# plt.legend(fontsize=10, loc='upper right')
ax.axis('off')
plt.show()
# plt.savefig('umap.png', dpi=300, bbox_inches='tight')

In [ ]:
spec = table[str(table['TARGETID'])[:1] == str(id)[:1]]
spec

In [ ]:
radius = 0.5

adj = radius_neighbors_graph(emb, radius=radius, include_self=False)

# 4) Etiquetas FoF
n_components, labels = connected_components(adj, directed=False)

# 5) Construcción del grafo
rows, cols = adj.nonzero()
G = nx.Graph()
G.add_edges_from(zip(rows, cols))

# 6) Posiciones y colormap
pos = {i: emb[i] for i in range(len(emb))}
cmap = ListedColormap(sns.color_palette("mako", n_components))

In [ ]:
nx.draw_networkx_edges(G, pos, alpha=0.3, width=0.1)
plt.axis('off')

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(9, 4), sharex=True, sharey=True)
nx.draw_networkx_edges(G, pos, alpha=0.5, width=0.1)

for i, c in enumerate(np.unique(cat_)):
    mask = np.where(cat_ == c)[0]
    axes[0].plot(emb[mask, 0], emb[mask, 1], 'o', markersize=4, alpha=0.8, label=c, color=colors[i])

axes[0].scatter(emb[outliers, 0], emb[outliers, 1], marker='o', s=100, alpha=0.6, label='outliers',
            c='firebrick', zorder=-1)
axes[1].scatter(emb[outliers, 0], emb[outliers, 1], marker='o', s=100, alpha=0.6, label='outliers',
            c='firebrick', zorder=-1)

axes[0].legend(fontsize=10, loc='upper left', ncol=1)
axes[1].legend(fontsize=10, loc='upper left', ncol=1)

axes[0].axis('off')
axes[1].axis('off')
plt.tight_layout()
plt.show()
# plt.savefig('umap.png', dpi=300, bbox_inches='tight')

# otro umap

In [ ]:
from umap import UMAP

In [ ]:
import sys, os
project_root = os.path.abspath('..')
sys.path.insert(0, project_root)

from src.desiproc.build_matrix import build_matrix
import glob

out_dir = os.path.join(project_root, 'AssessingDesiData', 'processed')
night   = '20211130'
tile    = '5568'
bands   = ['B','R','Z']
wg, fp, iv, z, ze, ids, cat, petals = build_matrix(out_dir, night, tile, bands)

print("wave_grid:", wg.shape)
print("flux matrix:", fp.shape)
print("ivar matrix:", iv.shape)
print("z vector:", z.shape)
print("zerr vector:", ze.shape)
print("ids vector:", ids.shape)
print("cat matrix:", cat.shape)
print("petals matrix:", petals.shape)

In [ ]:
defaults = dict(n_neighbors=100, min_dist=1., 
                n_components=2,
                        metric='cosine', n_jobs=-1)
reducer = UMAP(**defaults)

In [ ]:
X_emb = reducer.fit_transform(fp)

In [ ]:
radius = 0.5
adj = radius_neighbors_graph(X_emb, radius=radius, include_self=False)
n_components, labels = connected_components(adj, directed=False)

rows, cols = adj.nonzero()
G = nx.Graph()
G.add_edges_from(zip(rows, cols))

# 6) Posiciones y colormap
pos = {i: X_emb[i] for i in range(len(X_emb))}

In [ ]:
fig, ax = plt.subplots(1, 1)
nx.draw_networkx_edges(G, pos, alpha=0.3, width=0.1)

plt.plot(X_emb[:, 0], X_emb[:, 1], 'o', markersize=2, alpha=0.2)
plt.axis('off')
plt.show()

In [ ]:
graph = radius_neighbors_graph(X_emb, radius=1,
                                       mode='connectivity', include_self=True,
                                       n_jobs=-1)
n_clusters, labels = connected_components(csgraph=graph,
                                                    directed=False,
                                                    return_labels=True)

In [ ]:
uniq, cnt = np.unique(labels, return_counts=True)
small = uniq[cnt <= 50]
mask = np.isin(labels, small)

In [ ]:
X_emb[mask, 0]

In [ ]:
fig, ax = plt.subplots(1, 1)
nx.draw_networkx_edges(G, pos, alpha=0.3, width=0.1)

plt.plot(X_emb[mask, 0], X_emb[mask, 1], 'o', markersize=100, alpha=0.2)
plt.axis('off')
plt.show()